# 🐖 Notebook 18 — Tamaño y bootstrap por sesión nominal (CPU) | CORREGIDO

**Versión corregida:** se soluciona un defecto del análisis de AP por terciles.
La versión anterior modificaba `gt["ignore"]`; COCOeval sobrescribía ese indicador y producía el **mismo AP global en los tres grupos de tamaño**, lo cual era inválido.

Esta versión aplica **rangos de área relativa [área de máscara/(ancho × alto)]** mediante COCOeval, normalizados de forma consistente tanto para GT como para detecciones.
Los puntos de corte se derivan solo de GT de validación. Se verifica que el AP global permanece inalterado.

**Reutilización:** aprovecha `bootstrap_sesion.jsonl` del Notebook 18; no repite los 100 remuestreos ya completos, aunque recalcula el AP de los tamaños y los controles globales.

**Interpretación:** 13 prefijos de sesión nominales, no 13 sesiones independientemente verificadas. Los intervalos bootstrap siguen siendo exploratorios.

**Instrucciones:** CPU (sin GPU), ejecutar todas las celdas. No hay entrenamiento ni inferencia de nuevos modelos. Se guardará un respaldo del Excel anterior inválido.


## 1. Dependencias CPU

In [ ]:
%pip -q install "pycocotools>=2.0.8,<3" "pandas>=2,<3" numpy "openpyxl>=3.1" matplotlib

## 2. Rutas, fuentes y validación de datos

In [ ]:
from google.colab import drive,files
from pathlib import Path
from datetime import datetime,timezone
from collections import defaultdict
from copy import deepcopy
import json,zipfile,io,hashlib
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from contextlib import redirect_stdout

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir()
REPORT=ROOT/'03_Resultados'/'05_Reportes'; METRIC=ROOT/'03_Resultados'/'03_Metricas'
FIG=ROOT/'03_Resultados'/'04_Figuras';EXPORT=ROOT/'05_Exportaciones_ZIP'
for d in (REPORT,METRIC,FIG,EXPORT):d.mkdir(parents=True,exist_ok=True)
P09=REPORT/'CERDOS_09_COCO_Armonizado';P13=REPORT/'CERDOS_13_Calibracion_Validacion'
def rd(p):
 assert p.is_file(),f'No encuentro: {p}'
 return json.loads(p.read_text(encoding='utf8'))
GVAL=rd(P13/'CERDOS_13_GT_val_coco.json');GT=rd(P09/'CERDOS_09_GT_test_coco.json')
YP=rd(P09/'CERDOS_09_Pred_YOLO_coco.json');MP=rd(P09/'CERDOS_09_Pred_MaskRCNN_coco.json')
assert len(GVAL['images'])==196 and len(GT['images'])==373
assert len(GVAL['annotations'])==1995 and len(GT['annotations'])==4648
assert set(a['image_id'] for a in GT['annotations'])<=set(i['id'] for i in GT['images'])
PROTO={'version':'18_session_bootstrap_coco_v1','seed':2026,'repeticiones':100,
       'max_dets':100,'group':'filename_first_19_chars','tertial_quantiles_val':[1/3,2/3]}
RESULTDIR=REPORT/'CERDOS_18_Analisis_Sesion';RESULTDIR.mkdir(exist_ok=True)
PT=RESULTDIR/'protocolo.json'
if PT.exists():assert rd(PT)==PROTO,'La configuración cambió. No mezclar resultados.'
else:PT.write_text(json.dumps(PROTO,indent=2),encoding='utf8')
print('✅ Predicciones congeladas: YOLO',len(YP),'Mask',len(MP))


## 3. Motor COCO único y grupos temporales

In [ ]:
def make_coco(data):
 g=COCO();g.dataset=deepcopy(data)
 with redirect_stdout(io.StringIO()):g.createIndex()
 return g

def coco_ap(data,rows):
 g=make_coco(data)
 # No reutilizar objetos mutados por COCO.loadRes.
 pred=[{'image_id':int(x['image_id']),'category_id':int(x['category_id']),
        'segmentation':deepcopy(x['segmentation']),'score':float(x['score'])} for x in rows]
 if pred:
  with redirect_stdout(io.StringIO()):dt=g.loadRes(pred)
 else:
  dt=make_coco({'images':data['images'],'categories':data['categories'],
                'annotations':[],'info':{},'licenses':[]})
 ev=COCOeval(g,dt,'segm');ev.params.maxDets=[1,10,PROTO['max_dets']]
 with redirect_stdout(io.StringIO()):ev.evaluate();ev.accumulate();ev.summarize()
 return float(ev.stats[0])

imgs=GT['images'];imgid=set(int(i['id']) for i in imgs)
assert all(int(i['image_id']) in imgid for i in YP+MP)
sessions=defaultdict(list)
for im in imgs:sessions[im['file_name'][:19]].append(im)
assert len(sessions)>5,'Se esperaban varios prefijos nominales de sesión'
print('✅ Prefijos de sesión inferidos:',len(sessions),'en 5 fechas de test')
print('Tamaños de sesión',sorted([len(v) for v in sessions.values()],reverse=True)[:10])


## 4. Terciles GT normalizados y AP en cada tercil (experimental)

In [ ]:
# Evaluación CORREGIDA por terciles de área relativa. Los umbrales vienen SOLO
# de GT de validación; el test no interviene en la selección de rangos.
VIMG={int(i['id']):i for i in GVAL['images']}
relative_val=np.asarray([
    float(a['area'])/(VIMG[int(a['image_id'])]['height'] * VIMG[int(a['image_id'])]['width'])
    for a in GVAL['annotations']], dtype=float)
qlo,qhi=np.quantile(relative_val,[1/3,2/3])
assert 0 < qlo < qhi < 1
IM={int(i['id']):i for i in GT['images']}
SIZE_METHOD='COCOeval_relative_area_ranges_v2'
AREA_LABELS=['all','small','medium','large']
AREA_RANGES=[[0.0,1.000001],
             [0.0,float(qlo)],
             [float(np.nextafter(qlo,np.inf)),float(qhi)],
             [float(np.nextafter(qhi,np.inf)),1.000001]]
GT_SIZE=[]
for a in GT['annotations']:
    im=IM[int(a['image_id'])]
    frac=float(a['area'])/(int(im['width'])*int(im['height']))
    band='small' if frac<=qlo else 'medium' if frac<=qhi else 'large'
    GT_SIZE.append({'image_id':int(a['image_id']),'fecha':im['file_name'][:10],
                    'area_frac':frac,'tercil':band})
counts=pd.DataFrame(GT_SIZE).groupby('tercil').size().to_dict()
assert sum(counts.values())==len(GT['annotations'])
print('Terciles de área relativa ajustados en VALIDACIÓN:',qlo,qhi)
print('Número de instancias GT de TEST por tercil:',counts)

# No utilizar gt['ignore'] para filtrar tamaños: COCOeval._prepare() lo
# sobrescribe para segm con el estado de iscrowd, causando que todos los
# terciles produzcan exactamente el AP global.
# Usamos en su lugar los rangos de área nativos de COCOeval, en unidades
# relativas [área_máscara/(ancho*alto)] para GT y predicciones.
def coco_ap_per_relative_area(data,rows):
    normalized=deepcopy(data)
    dimensions={int(i['id']):int(i['width'])*int(i['height'])
                for i in normalized['images']}
    for a in normalized['annotations']:
        a['area']=float(a['area'])/dimensions[int(a['image_id'])]
    ground=make_coco(normalized)
    pred=[{'image_id':int(x['image_id']), 'category_id':int(x['category_id']),
           'segmentation':deepcopy(x['segmentation']), 'score':float(x['score'])}
          for x in rows]
    if pred:
        with redirect_stdout(io.StringIO()):det=ground.loadRes(pred)
        # loadRes calcula automáticamente el área en píxeles; COCOeval debe
        # recibir las mismas unidades relativas para GT y detecciones.
        for a in det.dataset['annotations']:
            a['area']=float(a['area'])/dimensions[int(a['image_id'])]
    else:
        det=make_coco({'images':normalized['images'],
                       'categories':normalized['categories'],
                       'annotations':[],'info':{},'licenses':[]})
    ev=COCOeval(ground,det,'segm')
    ev.params.maxDets=[1,10,PROTO['max_dets']]
    ev.params.areaRng=deepcopy(AREA_RANGES)
    ev.params.areaRngLbl=AREA_LABELS[:]
    with redirect_stdout(io.StringIO()):
        ev.evaluate();ev.accumulate();ev.summarize()
    # COCOeval.stats: 0 AP global, 3 AP small, 4 AP medium, 5 AP large
    return {'all':float(ev.stats[0]),
            'small':float(ev.stats[3]),
            'medium':float(ev.stats[4]),
            'large':float(ev.stats[5])}

SIZE_ROWS=[]
for model,rows in [('YOLO11n-seg',YP),('Mask R-CNN',MP)]:
    res=coco_ap_per_relative_area(GT,rows)
    ref=coco_ap(GT,rows)
    assert np.isfinite(res['all']) and abs(res['all']-ref)<1e-7, (model,res,ref)
    for band in ('small','medium','large'):
        SIZE_ROWS.append({'tercil_GT':band,'modelo':model,
                          'GT_tercil':int(counts.get(band,0)),
                          'AP_COCO_condicionado':res[band],
                          'area_rel_q33_val':float(qlo),
                          'area_rel_q67_val':float(qhi),
                          'metodo':SIZE_METHOD,
                          'mAP_COCO_global_control':res['all']})
SIZE=pd.DataFrame(SIZE_ROWS)
print('✅ AP estratificado con rangos de área relativa de COCOeval')
print('✅ Control: mAP global reproducido en ambos modelos')
display(SIZE)


## 5. Bootstrap apareado de sesiones nominales, con checkpoint

In [ ]:
# 100 remuestreos de prefijos de sesión; NO remuestreo ingenuo de imágenes.
# Puede tardar mucho: el checkpoint permite reanudar una ejecución parcial.
BFILE=RESULTDIR/'bootstrap_sesion.jsonl'
prev=[]
if BFILE.exists():
 for line in BFILE.read_text(encoding='utf8').splitlines():
  if line.strip():prev.append(json.loads(line))
assert [r['rep'] for r in prev]==list(range(len(prev))),'Checkpoint bootstrap incompleto/desordenado'

ann=defaultdict(list);preds={}
for a in GT['annotations']:ann[a['image_id']].append(a)
for model,rows in [('YOLO11n-seg',YP),('Mask R-CNN',MP)]:
 pp=defaultdict(list)
 for a in rows:pp[a['image_id']].append(a)
 preds[model]=pp
session_list=sorted(sessions.keys())

def sample_dataset(sampled,model):
 newims=[];newanns=[];newpred=[];ia=0
 for session in sampled:
  for src_img in sessions[session]:
   nid=len(newims)+1;im=dict(src_img);im['id']=nid;newims.append(im)
   for a in ann[src_img['id']]:
    ia+=1;b=dict(a);b['id']=ia;b['image_id']=nid;newanns.append(b)
   for a in preds[model][src_img['id']]:
    newpred.append({'image_id':nid,'category_id':int(a['category_id']),
                    'segmentation':a['segmentation'],'score':float(a['score'])})
 return {'info':{'description':'session-bootstrap'},'licenses':[],
         'categories':GT['categories'],'images':newims,'annotations':newanns},newpred

rg=np.random.default_rng(PROTO['seed'])
for rep in range(PROTO['repeticiones']):
 drawn=rg.choice(session_list,size=len(session_list),replace=True)
 if rep<len(prev):continue
 ap={}
 for model in ('YOLO11n-seg','Mask R-CNN'):
  ds,pr=sample_dataset(drawn,model)
  ap[model]=coco_ap(ds,pr)
 item={'rep':rep,'n_sesiones_muestreadas':len(drawn),'n_imagenes_en_bootstrap':len(ds['images']),
       'AP_YOLO':ap['YOLO11n-seg'],'AP_MaskRCNN':ap['Mask R-CNN'],
       'delta_AP_YOLO_menos_MaskRCNN':ap['YOLO11n-seg']-ap['Mask R-CNN']}
 with open(BFILE,'a',encoding='utf8') as f:f.write(json.dumps(item)+'\n')
 if (rep+1)%10==0:print('✅ Bootstrap sesiones',rep+1,'/',PROTO['repeticiones'])
print('✅ Remuestreos de sesión completos; no se cambió el test ni los pesos.')


## 6. Resumen, limitaciones, Excel, figura y ZIP

In [ ]:
BOOT=pd.DataFrame([json.loads(s) for s in BFILE.read_text(encoding='utf8').splitlines() if s.strip()])
assert len(BOOT)==PROTO['repeticiones']
vals=BOOT['delta_AP_YOLO_menos_MaskRCNN'].to_numpy(dtype=float)
GLOBAL=pd.DataFrame([{'modelo':'YOLO11n-seg','AP_COCO_global':coco_ap(GT,YP)},
                     {'modelo':'Mask R-CNN','AP_COCO_global':coco_ap(GT,MP)}])
D=GLOBAL.iloc[0]['AP_COCO_global']-GLOBAL.iloc[1]['AP_COCO_global']
B_SUM={'delta_AP_global':float(D),'delta_media_bootstrap':float(vals.mean()),
       'percentil_2_5_exploratorio':float(np.quantile(vals,.025)),
       'percentil_97_5_exploratorio':float(np.quantile(vals,.975)),
       'n_sesiones_nominales':len(sessions),'n_boot':len(BOOT)}
LIMITS=['Inferencia exploratoria con sesiones NOMINALES inferidas de prefijos de archivo.',
        'Tamaño de muestra de sesiones limitado; grupos pueden no ser independientes.',
        'Test fue consultado en fases 04 y 07 a 17, no hay evaluación externa nueva.',
        'Terciles se fijan usando GT de validación; no confundir con categorías COCO small/medium/large.',
        'AP por tercil usa rangos COCOeval de área normalizada definidos en validación; los AP por tamaño son condicionados.',
        'Intervalos porcentuales bootstrap no justifican afirmaciones causales ni entre granjas.']
X=METRIC/'CERDOS_18_Tamano_Bootstrap_Sesion.xlsx';J=REPORT/'CERDOS_18_Informe_Sesion.json'
if X.exists() and not (RESULTDIR/'respaldo_18_tamanos_invalidos.xlsx').exists():
 import shutil
 shutil.copy2(X,RESULTDIR/'respaldo_18_tamanos_invalidos.xlsx')
 print('Se conservó el Excel previo con terciles inválidos SOLO como registro histórico.')
with pd.ExcelWriter(X,engine='openpyxl') as w:
 GLOBAL.to_excel(w,sheet_name='mAP_COCO_Global',index=False)
 SIZE.to_excel(w,sheet_name='mAP_Terciles_GT',index=False)
 pd.DataFrame([{'metodo':SIZE_METHOD,'GT_area':'area_relativa',
                'pred_area':'area_relativa', 'q33_val':float(qlo), 'q67_val':float(qhi),
                'grupo':'prefijo 19 caracteres','remuestreos_bootstrap':PROTO['repeticiones']}]).to_excel(w,sheet_name='Metodo_tamanos',index=False)
 BOOT.to_excel(w,sheet_name='Bootstrap_sesiones',index=False)
 pd.DataFrame([B_SUM]).to_excel(w,sheet_name='Resumen_incertidumbre',index=False)
 pd.DataFrame({'advertencia':LIMITS}).to_excel(w,sheet_name='Limitaciones',index=False)
J.write_text(json.dumps({'protocolo':PROTO,'metodo_tamano':SIZE_METHOD,
                          'umbrales_relativos_val':[float(qlo),float(qhi)],
                          'n_GT_test_por_tercil':{k:int(v) for k,v in counts.items()},
                          'resumen':B_SUM,'limitaciones':LIMITS},
                         ensure_ascii=False,indent=2),encoding='utf8')
fig,ax=plt.subplots(figsize=(8,4.7))
ax.hist(vals,bins=min(18,max(5,int(np.sqrt(len(vals))))),color='#668EAF',edgecolor='white')
ax.axvline(D,color='#CD7E73',lw=2,label='Original paired ΔmAP')
ax.set(xlabel='YOLO − Mask R-CNN (segm mAP@50:95)',ylabel='Session bootstrap resamples',
       title='Exploratory session-grouped uncertainty')
ax.grid(alpha=.13);ax.legend();fig.tight_layout()
FIGURE=FIG/'CERDOS_18_Bootstrap_Sesiones.png';fig.savefig(FIGURE,dpi=400);plt.close(fig)
ZIP=EXPORT/'CERDOS_18_Tamano_Bootstrap_Sesion_Resultados.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for p in (X,J,FIGURE,PT):z.write(p,arcname=p.name)
assert zipfile.ZipFile(ZIP).testzip() is None
print('✅ RESULTADO exploratorio:',B_SUM,'\nZIP:',ZIP)
files.download(str(ZIP))
